# Stage 1 – Three‑Case Comparison (Baseline vs Seasonal‑Off vs Seasonal‑On)

This notebook compares three runs:
- baseline (no seasonal)
- seasonal storage added but parameters **off** (acts like additional battery)
- seasonal storage parameters **on**

It reports:
- equivalence check: baseline vs seasonal‑off
- seasonal usage and timing
- shortage (load shedding) events and reliability stress
- renewable curtailment vs storage activity


In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")


In [ ]:
import re
pattern = re.compile(r"(\\d{4})$")
# --- User configuration -----------------------------------------------------
PROJECT_DIR = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN_NAME = "low_RE_mod_elec_iter0"
CLIMATE_SCENARIO = "historical_1980_2019"

RUNS = {
    "baseline": "stagel_3month_base",
    "pcap2160_1cycle": "stagel_3month_1cycle",
    "pcap720_1cycle": "stagel_1month_1cycle",
}

# set YEAR to None to auto-detect
YEAR = 1985

output_root = PROJECT_DIR / "runs" / RUN_NAME / "outputs" / CLIMATE_SCENARIO


def detect_years(save_name: str):
    run_dir = output_root / save_name
    years = set()
    for p in run_dir.glob("charge_*.csv"):
        m = pattern.search(p.stem)
        if m:
            years.add(int(m.group(1)))
    return sorted(years)

if YEAR is None:
    years = set()
    for name in RUNS.values():
        years |= set(detect_years(name))
    YEARS = sorted(years)
else:
    YEARS = [YEAR]

print("Runs:", RUNS)
print("Years:", YEARS)


In [ ]:
# --- Helpers ----------------------------------------------------------------

def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def tidy_bus_csv(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def load_storage_csv(run_dir: Path, year: int, value_name: str, split: str | None = None,
                     fallback_combined: bool = True) -> pd.DataFrame:
    if split in {"base", "seasonal"}:
        split_path = run_dir / f"{value_name}_{split}_{year}.csv"
        if split_path.exists():
            df = pd.read_csv(split_path)
            df = df.copy()
            df["asset_type"] = split
            df["is_seasonal"] = 1 if split == "seasonal" else 0
            return tidy_storage_df(df, value_name)
        if not fallback_combined:
            raise FileNotFoundError(split_path)

    df = pd.read_csv(run_dir / f"{value_name}_{year}.csv")
    debug_path = run_dir / "_storage_debug.csv"
    if debug_path.exists():
        debug = pd.read_csv(debug_path)
        if "is_seasonal" in debug.columns and len(debug) == len(df):
            df = df.copy()
            if "asset_type" not in df.columns:
                df["asset_type"] = "base"
            df.loc[debug["is_seasonal"] == 1, "asset_type"] = "seasonal"
            df["is_seasonal"] = debug["is_seasonal"].values

    if split in {"base", "seasonal"} and "asset_type" in df.columns:
        df = df[df["asset_type"] == split].copy()

    return tidy_storage_df(df, value_name)


def total_timeseries(df: pd.DataFrame, value_name: str) -> pd.Series:
    return df.groupby("timestamp")[value_name].sum()


def event_stats(series: pd.Series, threshold: float = 1e-6) -> pd.DataFrame:
    s = series.fillna(0.0)
    events = []
    in_event = False
    start = None
    total = 0.0
    hours = 0
    for ts, val in s.items():
        if val > threshold:
            if not in_event:
                in_event = True
                start = ts
                total = 0.0
                hours = 0
            total += float(val)
            hours += 1
            end = ts
        else:
            if in_event:
                events.append((start, end, hours, total))
                in_event = False
    if in_event:
        events.append((start, end, hours, total))

    if not events:
        return pd.DataFrame(columns=["start", "end", "hours", "total_MWh"])
    df = pd.DataFrame(events, columns=["start", "end", "hours", "total_MWh"])
    return df.sort_values("hours", ascending=False)


In [ ]:
# --- Load runs --------------------------------------------------------------
run_data = {label: {} for label in RUNS}

for label, save_name in RUNS.items():
    run_dir = output_root / save_name
    for year in YEARS:
        charge = load_storage_csv(run_dir, year, "charge")
        discharge = load_storage_csv(run_dir, year, "discharge")
        charge_base = load_storage_csv(run_dir, year, "charge", split="base")
        discharge_base = load_storage_csv(run_dir, year, "discharge", split="base")
        charge_seasonal = load_storage_csv(run_dir, year, "charge", split="seasonal")
        discharge_seasonal = load_storage_csv(run_dir, year, "discharge", split="seasonal")

        load_shed = tidy_bus_csv(run_dir / f"load_shedding_{year}.csv", "load_shedding")
        wind_curt = tidy_bus_csv(run_dir / f"wind_curtailment_{year}.csv", "wind_curtailment")
        solar_curt = tidy_bus_csv(run_dir / f"solar_curtailment_{year}.csv", "solar_curtailment")

        run_data[label][year] = {
            "charge": charge,
            "discharge": discharge,
            "charge_base": charge_base,
            "discharge_base": discharge_base,
            "charge_seasonal": charge_seasonal,
            "discharge_seasonal": discharge_seasonal,
            "load_shed": load_shed,
            "wind_curt": wind_curt,
            "solar_curt": solar_curt,
        }

print("Loaded runs:")
for label in RUNS:
    print(f"  {label}: years -> {sorted(run_data[label].keys())}")


In [ ]:
# --- Baseline vs Scenario A/B: storage activity ----------------------------
scenario_labels = ["pcap2160_1cycle", "pcap720_1cycle"]

for sc in scenario_labels:
    for year in YEARS:
        base = run_data["baseline"][year]
        run = run_data[sc][year]

        fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
        # base
        base_ch = total_timeseries(base["charge_base"], "charge")
        base_dis = total_timeseries(base["discharge_base"], "discharge")
        base_seas_ch = total_timeseries(base["charge_seasonal"], "charge")
        base_seas_dis = total_timeseries(base["discharge_seasonal"], "discharge")

        # scenario
        sc_ch = total_timeseries(run["charge_base"], "charge")
        sc_dis = total_timeseries(run["discharge_base"], "discharge")
        sc_seas_ch = total_timeseries(run["charge_seasonal"], "charge")
        sc_seas_dis = total_timeseries(run["discharge_seasonal"], "discharge")

        axes[0].plot(base_ch.index, base_ch.values, label="baseline base charge", alpha=0.6)
        axes[0].plot(base_dis.index, -base_dis.values, label="baseline base discharge", alpha=0.6, linestyle="--")
        axes[0].plot(base_seas_ch.index, base_seas_ch.values, label="baseline seasonal charge", alpha=0.6)
        axes[0].plot(base_seas_dis.index, -base_seas_dis.values, label="baseline seasonal discharge", alpha=0.6, linestyle=":")
        axes[0].axhline(0, color="black", linewidth=0.8)
        axes[0].set_title(f"Baseline storage activity ({year})")
        axes[0].legend(ncol=2, fontsize=8)

        axes[1].plot(sc_ch.index, sc_ch.values, label=f"{sc} base charge", alpha=0.6)
        axes[1].plot(sc_dis.index, -sc_dis.values, label=f"{sc} base discharge", alpha=0.6, linestyle="--")
        axes[1].plot(sc_seas_ch.index, sc_seas_ch.values, label=f"{sc} seasonal charge", alpha=0.6)
        axes[1].plot(sc_seas_dis.index, -sc_seas_dis.values, label=f"{sc} seasonal discharge", alpha=0.6, linestyle=":")
        axes[1].axhline(0, color="black", linewidth=0.8)
        axes[1].set_title(f"{sc} storage activity ({year})")
        axes[1].legend(ncol=2, fontsize=8)

        plt.tight_layout()
        plt.show()


In [ ]:
# --- Baseline vs Scenario A/B: load shedding ------------------------------
scenario_labels = ["pcap2160_1cycle", "pcap720_1cycle"]

for sc in scenario_labels:
    for year in YEARS:
        base = run_data["baseline"][year]
        run = run_data[sc][year]

        base_ls = total_timeseries(base["load_shed"], "load_shedding")
        sc_ls = total_timeseries(run["load_shed"], "load_shedding")

        fig, ax = plt.subplots(1, 1, figsize=(14, 4))
        ax.plot(base_ls.index, base_ls.values, label="baseline load shed", alpha=0.7)
        ax.plot(sc_ls.index, sc_ls.values, label=f"{sc} load shed", alpha=0.7)
        ax.set_title(f"Load shedding comparison ({year})")
        ax.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
# --- Baseline vs Scenario A/B: curtailment --------------------------------
scenario_labels = ["pcap2160_1cycle", "pcap720_1cycle"]

for sc in scenario_labels:
    for year in YEARS:
        base = run_data["baseline"][year]
        run = run_data[sc][year]

        base_w = total_timeseries(base["wind_curt"], "wind_curtailment")
        base_s = total_timeseries(base["solar_curt"], "solar_curtailment")
        sc_w = total_timeseries(run["wind_curt"], "wind_curtailment")
        sc_s = total_timeseries(run["solar_curt"], "solar_curtailment")

        fig, ax = plt.subplots(1, 1, figsize=(14, 4))
        ax.plot(base_w.index, base_w.values, label="baseline wind curt", alpha=0.6)
        ax.plot(base_s.index, base_s.values, label="baseline solar curt", alpha=0.6)
        ax.plot(sc_w.index, sc_w.values, label=f"{sc} wind curt", alpha=0.6)
        ax.plot(sc_s.index, sc_s.values, label=f"{sc} solar curt", alpha=0.6)
        ax.set_title(f"Curtailment comparison ({year})")
        ax.legend(ncol=2, fontsize=8)
        plt.tight_layout()
        plt.show()


In [ ]:
# --- Sanity check ------------------------------------------------------------
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

print("YEARS:", YEARS)
if not YEARS:
    raise ValueError("YEARS is empty. Check RUNS names and output files.")

for label in RUNS:
    year = YEARS[0]
    rd = run_data[label][year]
    print(f"{label}: charge rows = {len(rd['charge'])}, discharge rows = {len(rd['discharge'])}")


## 1) Baseline vs Scenario A/B (Storage Activity)


In [ ]:
for year in YEARS:
    base = run_data["baseline"][year]
    off = run_data["seasonal_off"][year]

    base_ls = total_timeseries(base["load_shed"], "load_shedding")
    off_ls = total_timeseries(off["load_shed"], "load_shedding")

    base_dis = total_timeseries(base["discharge"], "discharge")
    off_dis = total_timeseries(off["discharge"], "discharge")

    base_charge = total_timeseries(base["charge"], "charge")
    off_charge = total_timeseries(off["charge"], "charge")

    # Align indices
    idx = base_ls.index.union(off_ls.index)
    base_ls = base_ls.reindex(idx, fill_value=0.0)
    off_ls = off_ls.reindex(idx, fill_value=0.0)
    base_dis = base_dis.reindex(idx, fill_value=0.0)
    off_dis = off_dis.reindex(idx, fill_value=0.0)
    base_charge = base_charge.reindex(idx, fill_value=0.0)
    off_charge = off_charge.reindex(idx, fill_value=0.0)

    diff = {
        "load_shed_MWh_diff": float((off_ls - base_ls).sum()),
        "discharge_MWh_diff": float((off_dis - base_dis).sum()),
        "charge_MWh_diff": float((off_charge - base_charge).sum()),
        "max_abs_load_shed_MW": float((off_ls - base_ls).abs().max()),
        "max_abs_discharge_MW": float((off_dis - base_dis).abs().max()),
        "max_abs_charge_MW": float((off_charge - base_charge).abs().max()),
    }

    print(f"\nYear {year} – baseline vs seasonal_off")
    display(pd.DataFrame([diff]))

    # Plot overlay
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(base_dis.index, base_dis.values, label="baseline discharge", alpha=0.7)
    axes[0].plot(off_dis.index, off_dis.values, label="seasonal_off discharge", alpha=0.7)
    axes[0].set_title("Total discharge comparison")
    axes[0].legend()

    axes[1].plot(base_ls.index, base_ls.values, label="baseline load shed", alpha=0.7)
    axes[1].plot(off_ls.index, off_ls.values, label="seasonal_off load shed", alpha=0.7)
    axes[1].set_title("Load shedding comparison")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 2) Baseline vs Scenario A/B (Load Shedding)


In [ ]:
for year in YEARS:
    run = run_data["seasonal_on"][year]
    base_dis = total_timeseries(run["discharge_base"], "discharge")
    seasonal_dis = total_timeseries(run["discharge_seasonal"], "discharge")
    base_ch = total_timeseries(run["charge_base"], "charge")
    seasonal_ch = total_timeseries(run["charge_seasonal"], "charge")
    ls = total_timeseries(run["load_shed"], "load_shedding")

    print(f"\nYear {year} – seasonal_on usage")
    print(f"  base_discharge_MWh: {base_dis.sum():.1f}")
    print(f"  seasonal_discharge_MWh: {seasonal_dis.sum():.1f}")
    print(f"  seasonal_discharge_hours: {int((seasonal_dis>0).sum())}")
    print(f"  load_shed_hours: {int((ls>0).sum())}")

    # Weekly aggregation to see seasonal usage timing
    # Normalize timezone to avoid tz-aware vs tz-naive joins
    for s in [base_dis, seasonal_dis, ls, base_ch, seasonal_ch]:
        if hasattr(s.index, 'tz') and s.index.tz is not None:
            s.index = s.index.tz_convert(None)
    weekly = pd.DataFrame({
        "seasonal_discharge_MWh": seasonal_dis.resample("W").sum(),
        "base_discharge_MWh": base_dis.resample("W").sum(),
        "load_shed_MWh": ls.resample("W").sum(),
    }).fillna(0.0)

    display(weekly.head(10))

    # Plot seasonal vs base + load shed
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(base_ch.index, base_ch.values, label="base charge", alpha=0.6)
    axes[0].plot(base_dis.index, -base_dis.values, label="base discharge", alpha=0.6)
    axes[0].plot(seasonal_ch.index, seasonal_ch.values, label="seasonal charge", alpha=0.6)
    axes[0].plot(seasonal_dis.index, -seasonal_dis.values, label="seasonal discharge", alpha=0.6)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title("Storage fleet activity (seasonal_on)")
    axes[0].legend(loc="upper right")

    axes[1].plot(ls.index, ls.values, color="C3", label="load shedding")
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Load shedding (seasonal_on)")
    axes[1].legend(loc="upper right")

    plt.tight_layout()
    plt.show()


## 3) Long Shortage Events and Reliability Stress

In [ ]:
for year in YEARS:
    run = run_data["seasonal_on"][year]
    ls = total_timeseries(run["load_shed"], "load_shedding")
    events = event_stats(ls, threshold=0.0)

    print(f"
Year {year} – load shedding events")
    display(events.head(10))

    # Convert hours to weeks for top events
    if not events.empty:
        events_weeks = events.copy()
        events_weeks["weeks"] = events_weeks["hours"] / (24*7)
        display(events_weeks.head(10))

    print("Total load shedding hours:", int((ls > 0).sum()))
    print("Peak load shedding MW:", float(ls.max()))


## 3) Baseline vs Scenario A/B (Curtailment)


In [ ]:
for year in YEARS:
    run = run_data["seasonal_on"][year]
    wind = total_timeseries(run["wind_curt"], "wind_curtailment")
    solar = total_timeseries(run["solar_curt"], "solar_curtailment")
    base_ch = total_timeseries(run["charge_base"], "charge")
    seasonal_ch = total_timeseries(run["charge_seasonal"], "charge")

    fig, ax = plt.subplots(1, 1, figsize=(14, 4))
    ax.plot(wind.index, wind.values, label="wind curtailment", alpha=0.7)
    ax.plot(solar.index, solar.values, label="solar curtailment", alpha=0.7)
    ax.plot(base_ch.index, base_ch.values, label="base charge", alpha=0.7)
    ax.plot(seasonal_ch.index, seasonal_ch.values, label="seasonal charge", alpha=0.7)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Curtailment vs charging (seasonal_on)")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
